# Spectrogram Model Training Pipeline

This notebook processes data and trains a spectrogram-based model for chorus detection.

In [1]:
# Setup and imports
import os
import sys
import pandas as pd
import numpy as np
import gzip
from pathlib import Path
import pickle
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import Adam, SGD, AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau, OneCycleLR
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import librosa
import librosa.display
from sklearn.decomposition import NMF
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from sklearn.preprocessing import StandardScaler
import random
from torch.nn.parallel import DistributedDataParallel
from torch.utils.tensorboard import SummaryWriter
import ray
from ray import tune
from ray.tune import CLIReporter
from ray.tune.schedulers import ASHAScheduler, PopulationBasedTraining
from functools import partial

# Add the project root to the Python path
sys.path.append(os.path.abspath('..'))


## Paths and Configuration

In [4]:
experiment = "spec"
version = "1"

csv_file = "../data/dataframes/labels.csv"
audio_dir = "../data/audio/processed"
output_dir = "../models/spectrogram"
labels_dir = f"../data/labels/{experiment}/v{version}"
precomputed_dir = "../data/precomputed"

# Ensure directories exist
os.makedirs(labels_dir, exist_ok=True)
os.makedirs(precomputed_dir, exist_ok=True)


df = pd.read_csv(csv_file)
target_sr=12000
hop_length=128

Preprocess melspectrogram features

In [ ]:
# Parameters/paths
df = pd.read_csv(csv_file)
audio_dir = '../data/audio/processed'
output_dir = '../data/features/melspectrogram_segments'
sr = 12000
hop_length = 128
n_mels = 128
n_components = 3

all_segments = {}

# Process each song
for song_id in tqdm(df['song_id'].unique(), desc="Processing songs"):
    # Load audio
    audio_path = os.path.join(audio_dir, f'{song_id}.mp3')
    y, _ = librosa.load(audio_path, sr=sr)
    
    # Compute mel spectrogram
    mel_spec = librosa.feature.melspectrogram(
        y=y,
        sr=sr,
        n_mels=n_mels,
        hop_length=hop_length
    )
    
    # Convert to log power mel spectrogram
    log_mel = librosa.power_to_db(mel_spec, ref=np.max)
    
    # Normalize
    normalized_mel = (log_mel - log_mel.mean()) / (log_mel.std() + 1e-6)
    
    # Ensure non-negativity for NMF
    positive_mel = normalized_mel - normalized_mel.min() + 1e-6
    
    # Apply NMF
    model = NMF(n_components=n_components, init='nndsvda', random_state=42, max_iter=500)
    W = model.fit_transform(positive_mel.T)  # shape: (frames, n_components)
    H = model.components_  # shape: (n_components, n_mels)

    # Find peak frequency (mel band) for each component
    peak_freqs = np.argmax(H, axis=1)
    sort_idx = np.argsort(peak_freqs)  # Sort by ascending peak frequency

    # Reorder W and H
    W = W[:, sort_idx]
    H = H[sort_idx]
    
    # Get song metadata
    song_data = df[df['song_id'] == song_id].iloc[0]
    
    # Calculate tempo and beats
    tempo, beats = librosa.beat.beat_track(y=y, sr=sr, hop_length=hop_length)
    bpm = np.clip(tempo, 70, 140)
    
    # Get time signature (default to 4 if not available)
    time_signature = song_data.get('sp_time_signature', 4)
    if pd.isna(time_signature) or time_signature == 0:
        time_signature = 4
    time_signature = int(time_signature)
    
    # Create meter grid
    first_beat_time = librosa.frames_to_time(beats[0], sr=sr, hop_length=hop_length)
    time_duration = librosa.frames_to_time(W.shape[0], sr=sr, hop_length=hop_length)
    seconds_per_beat = 60.0 / bpm
    num_beats_forward = int((time_duration - first_beat_time) / seconds_per_beat)
    num_beats_backward = int(first_beat_time / seconds_per_beat) + 1
    
    beat_times_forward = first_beat_time + np.arange(num_beats_forward) * seconds_per_beat
    beat_times_backward = first_beat_time - np.arange(1, num_beats_backward) * seconds_per_beat
    beat_grid = np.concatenate((np.array([0.0]), beat_times_backward[::-1], beat_times_forward))
    
    meter_indices = np.arange(0, len(beat_grid), time_signature)
    meter_grid = beat_grid[meter_indices]
    
    if meter_grid[0] != 0.0:
        meter_grid = np.insert(meter_grid, 0, 0.0)
    
    meter_grid = librosa.time_to_frames(meter_grid, sr=sr, hop_length=hop_length)
    
    if meter_grid[-1] != W.shape[0]:
        meter_grid = np.append(meter_grid, W.shape[0])
    
    # Segment using meter grid
    meter_segments = []
    for i in range(len(meter_grid) - 1):
        start = meter_grid[i]
        end = meter_grid[i + 1]
        if start < W.shape[0] and end <= W.shape[0]:
            segment = W[start:end]
            meter_segments.append(segment)
    
    # Save segments
    if meter_segments:
        with gzip.open(os.path.join(output_dir, f'{song_id}_segments.pkl.gz'), 'wb') as f:
            pickle.dump(meter_segments, f)

Processing songs:   0%|          | 0/331 [00:00<?, ?it/s]

c:\Users\denni\anaconda3\envs\chorus-detection\lib\site-packages\sklearn\decomposition\_nmf.py:1710: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
c:\Users\denni\anaconda3\envs\chorus-detection\lib\site-packages\sklearn\decomposition\_nmf.py:1710: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
c:\Users\denni\anaconda3\envs\chorus-detection\lib\site-packages\sklearn\decomposition\_nmf.py:1710: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
c:\Users\denni\anaconda3\envs\chorus-detection\lib\site-packages\sklearn\decomposition\_nmf.py:1710: ConvergenceWarning: Maximum number of iterations 500 reached. Increase it to improve convergence.
  warnings.warn(
c:\Users\denni\anaconda3\envs\chorus-detection\lib\site-packages\sklearn\decomposition\_nmf.py:1710: ConvergenceWarning: Maximum number 

Preprocess labels

In [ ]:
df = pd.read_csv(csv_file)
target_sr=12000
hop_length=128


# Calculate frame rate for 12kHz
frame_rate_12khz = target_sr / hop_length
print(f"Frame rate at {target_sr}Hz: {frame_rate_12khz:.1f} frames/second")

os.makedirs(labels_dir, exist_ok=True)

processed_songs = 0

for song_id, group in df.groupby('song_id'):
    # Convert time to frames at 12kHz
    max_time = group['end_time'].max()
    max_frame_12khz = int(max_time * target_sr / hop_length)
    
    labels = np.zeros(max_frame_12khz, dtype=np.uint8)
    
    for _, row in group.iterrows():
        if row['label'] == 'chorus':
            start_frame_12khz = int(row['start_time'] * target_sr / hop_length)
            end_frame_12khz = int(row['end_time'] * target_sr / hop_length)
            
            # Ensure frames are within bounds
            start_frame_12khz = max(0, start_frame_12khz)
            end_frame_12khz = min(max_frame_12khz, end_frame_12khz)
            
            if start_frame_12khz < end_frame_12khz:
                labels[start_frame_12khz:end_frame_12khz] = 1
    
    filename = f"{song_id}_labels_{experiment}_v{version}.pkl.gz"
    filepath = os.path.join(labels_dir, filename)
    
    with gzip.open(filepath, 'wb') as f:
        pickle.dump(labels, f, protocol=pickle.HIGHEST_PROTOCOL)
    
    processed_songs += 1

Frame rate at 12000Hz: 93.8 frames/second


Calculate max frames and meters throughout entire dataset

In [7]:
# Calculate max frames and meters
segments_dir = '../data/features/melspectrogram_segments'
max_frames_per_meter = 0
max_meters = 0

for filename in tqdm(os.listdir(segments_dir), desc="Processing saved segments"):
    if filename.endswith('.pkl.gz'):
        with gzip.open(os.path.join(segments_dir, filename), 'rb') as f:
            segments = pickle.load(f)
            max_meters = max(max_meters, len(segments))
            max_frames_per_meter = max(max_frames_per_meter, 
                                     max(segment.shape[0] for segment in segments))

print(f"Max frames per meter: {max_frames_per_meter}")
print(f"Max meters per song: {max_meters}")

Processing saved segments:   0%|          | 0/331 [00:00<?, ?it/s]

Max frames per meter: 348
Max meters per song: 203


Define Dataset Class

In [9]:
class MelDecompDataset(Dataset):
    def __init__(self, audio_files, labels_dir, config, precomputed_dir=None, transform=None, 
                 cache_data=True, n_components=8, experiment="spec", version="1"):
        self.audio_files = audio_files
        self.song_ids = [os.path.splitext(os.path.basename(f))[0] for f in audio_files]
        self.labels_dir = labels_dir
        self.precomputed_dir = precomputed_dir
        self.transform = transform
        self.cache_data = cache_data
        self.n_components = n_components
        self.max_frames = config["data"]["max_frames"]
        self.max_meters = config["data"]["max_meters"]
        self.hop_length = config["audio"]["hop_length"]
        self.sr = config["audio"]["sr"]
        self.experiment = experiment
        self.version = version
        
        # Cache for data to avoid reloading
        self.data_cache = {}
    
    def _extract_mel_components(self, audio_file):
        # Get song_id from the audio file path
        song_id = os.path.splitext(os.path.basename(audio_file))[0]
        
        # Load audio with librosa
        y, sr = librosa.load(audio_file, sr=self.sr)
        
        # Compute melspectrogram
        mel_spec = librosa.feature.melspectrogram(
            y=y, sr=sr, hop_length=self.hop_length, n_mels=80
        )
        
        # Convert to log scale
        log_mel = librosa.power_to_db(mel_spec, ref=np.max)
        
        # Normalize
        normalized_mel = (log_mel - log_mel.mean()) / (log_mel.std() + 1e-6)
        
        # Apply NMF to decompose the melspectrogram
        model = NMF(n_components=self.n_components, init='nndsvda', random_state=42, l1_ratio=0.8, max_iter=500)
        
        # Ensure non-negative input for NMF
        positive_mel = normalized_mel - normalized_mel.min() + 1e-6
        
        # Fit model with warning capture
        import warnings
        with warnings.catch_warnings(record=True) as w:
            warnings.simplefilter("always")
            W = model.fit_transform(positive_mel.T)  # components activation over time
            
            # Check if convergence warning occurred
            if w and any("Maximum number of iterations" in str(warning.message) for warning in w):
                print(f"⚠️  CONVERGENCE WARNING for song_id: {song_id}")
                # Optionally log to file
                with open("convergence_warnings.log", "a") as f:
                    f.write(f"{song_id}\n")
        
        return y, sr, W
    
    def _create_meter_grid(self, y, sr):
        # Determine beat positions using librosa
        tempo, beats = librosa.beat.beat_track(y=y, sr=sr)
        
        # Convert beat positions to spectrogram frames
        beat_frames = librosa.time_to_frames(
            librosa.frames_to_time(beats), sr=sr, hop_length=self.hop_length
        )
        
        # Group beats into meters (assuming 4 beats per meter)
        beats_per_meter = 4
        meter_frames = [beat_frames[i:i+beats_per_meter] for i in range(0, len(beat_frames), beats_per_meter)]
        
        # Create meter boundaries
        meter_boundaries = []
        prev_end = 0
        
        for meter in meter_frames:
            if len(meter) > 0:
                start = prev_end
                end = meter[-1] if len(meter) > 0 else prev_end
                meter_boundaries.append((start, end))
                prev_end = end
        
        return meter_boundaries
    
    def _segment_data_meters(self, features, meter_grid):
        segments = []
        
        for start, end in meter_grid:
            # Ensure we have valid indices
            if start < features.shape[0] and end < features.shape[0]:
                segment = features[start:end+1]
                segments.append(segment)
        
        return segments
    
    def _pad_song(self, segments):
        padded_song = np.zeros((self.max_meters, self.max_frames, self.n_components))
        
        for i, segment in enumerate(segments):
            if i >= self.max_meters:
                break
                
            segment_frames = segment.shape[0]
            
            if segment_frames <= self.max_frames:
                # Pad with zeros if too short
                padded_song[i, :segment_frames, :] = segment
            else:
                # If segment is too long, sample frames evenly
                indices = np.linspace(0, segment_frames - 1, self.max_frames, dtype=int)
                padded_song[i, :, :] = segment[indices, :]
                
        return padded_song
    
    def _pad_labels(self, labels):
        if len(labels) > self.max_meters:
            return labels[:self.max_meters].reshape(-1, 1)
        else:
            padded = np.pad(
                labels, 
                (0, self.max_meters - len(labels)), 
                mode='constant', 
                constant_values=-1
            )
            return padded.reshape(-1, 1)
    
    def _load_item(self, idx):
        song_id = self.song_ids[idx]
        
        # Check if already in cache
        if song_id in self.data_cache:
            return self.data_cache[song_id]
        
        # Check if precomputed data exists
        if self.precomputed_dir:
            features_path = os.path.join(self.precomputed_dir, f"{song_id}_mel_decomp_features.pkl")
            if os.path.exists(features_path):
                try:
                    with open(features_path, "rb") as f:
                        padded_song = pickle.load(f)
                        
                    # Load labels
                    labels_path = os.path.join(self.labels_dir, f"{song_id}_labels_{self.experiment}_v{self.version}.pkl.gz")
                    with gzip.open(labels_path, "rb") as f:
                        labels = pickle.load(f)
                    
                    padded_labels = self._pad_labels(labels)
                    
                    if self.cache_data:
                        self.data_cache[song_id] = (padded_song, padded_labels)
                        
                    return padded_song, padded_labels
                    
                except Exception as e:
                    print(f"Error loading precomputed data for {song_id}: {e}")
        
        # Process audio if not precomputed
        try:
            audio_file = self.audio_files[idx]
            
            # Extract melspectrogram and components directly with librosa
            y, sr, mel_components = self._extract_mel_components(audio_file)
            
            # Create meter grid
            meter_grid = self._create_meter_grid(y, sr)
            
            # Transpose mel_components to have shape [frames, components]
            features = mel_components
            
            # Segment by meters
            segments = self._segment_data_meters(features, meter_grid)
            
            # Pad to create uniform 3D array
            padded_song = self._pad_song(segments)
            
            # Load labels
            labels_path = os.path.join(self.labels_dir, f"{song_id}_labels_{self.experiment}_v{self.version}.pkl.gz")
            with gzip.open(labels_path, "rb") as f:
                labels = pickle.load(f)
            
            padded_labels = self._pad_labels(labels)
            
            # Save precomputed features if directory specified
            if self.precomputed_dir:
                os.makedirs(self.precomputed_dir, exist_ok=True)
                with open(os.path.join(self.precomputed_dir, f"{song_id}_mel_decomp_features.pkl"), "wb") as f:
                    pickle.dump(padded_song, f)
            
            # Cache result
            if self.cache_data:
                self.data_cache[song_id] = (padded_song, padded_labels)
                
            return padded_song, padded_labels
            
        except Exception as e:
            print(f"Error processing {song_id}: {e}")
            # Return a zero tensor as fallback
            padded_song = np.zeros((self.max_meters, self.max_frames, self.n_components))
            padded_labels = np.ones((self.max_meters, 1)) * -1  # All masked
            
            if self.cache_data:
                self.data_cache[song_id] = (padded_song, padded_labels)
                
            return padded_song, padded_labels
    
    def __len__(self):
        return len(self.audio_files)
    
    def __getitem__(self, idx):
        # Load data
        features, labels = self._load_item(idx)
        
        # Convert to torch tensors
        features_tensor = torch.tensor(features, dtype=torch.float32)
        labels_tensor = torch.tensor(labels, dtype=torch.float32)
        
        # Apply transform if specified
        if self.transform:
            features_tensor = self.transform(features_tensor)
            
        return features_tensor, labels_tensor

Define Model Class

In [10]:
class MelDecompModel(nn.Module):
    def __init__(self, config):
        super(MelDecompModel, self).__init__()
        self.config = config
        
        # Data parameters
        self.max_frames = config["data"]["max_frames"]
        self.max_meters = config["data"]["max_meters"]
        self.n_components = config.get("audio", {}).get("n_components", 3)
        self.dropout_rate = config["model"]["dropout"]
        
        # CNN layers - FIXED pooling strategy
        cnn_config = config["model"]["cnn"]
        self.conv_layers = nn.ModuleList()
        self.pool_layers = nn.ModuleList()
        self.norm_layers = nn.ModuleList()
        
        # Create CNN layers
        in_channels = 1
        channels = cnn_config["channels"]
        kernel_sizes = cnn_config["kernel_sizes"]
        pool_sizes = cnn_config["pool_sizes"]
        
        for i in range(len(channels)):
            self.conv_layers.append(
                nn.Conv2d(
                    in_channels=in_channels,
                    out_channels=channels[i],
                    kernel_size=kernel_sizes[i],
                    padding='same'
                )
            )
            self.norm_layers.append(nn.BatchNorm2d(channels[i]))
            self.pool_layers.append(nn.MaxPool2d(kernel_size=pool_sizes[i]))
            in_channels = channels[i]
        
        # Calculate output size - FIXED calculation
        # Input: (batch, 1, n_components, max_frames) = (batch, 1, 3, 300)
        feat_dim = self.n_components  # 3
        time_dim = self.max_frames    # 300
        
        print(f"Initial: feat_dim={feat_dim}, time_dim={time_dim}")
        
        for i, pool_size in enumerate(pool_sizes):
            if isinstance(pool_size, tuple):
                feat_dim = max(1, feat_dim // pool_size[0])
                time_dim = max(1, time_dim // pool_size[1])
            else:
                feat_dim = max(1, feat_dim // pool_size)
                time_dim = max(1, time_dim // pool_size)
            print(f"After pool {i}: feat_dim={feat_dim}, time_dim={time_dim}")
        
        cnn_output_size = feat_dim * time_dim * channels[-1]
        print(f"CNN output size: {cnn_output_size}")
        
        # Ensure valid output size
        if cnn_output_size <= 0:
            raise ValueError(f"CNN output size is {cnn_output_size}. Adjust pooling strategy.")
        
        # Meter-level processing
        self.meter_layer = nn.Sequential(
            nn.Linear(cnn_output_size, 128),
            nn.ReLU(),
            nn.Dropout(self.dropout_rate),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(self.dropout_rate)
        )
        
        # Output layer
        self.output_layer = nn.Linear(64, 1)
        
        # Initialize weights
        self._initialize_weights()
    
    def forward(self, x):
        # x shape: (batch_size, max_meters, max_frames, n_components)
        batch_size = x.size(0)
        outputs = []
        
        # Process each meter separately
        for meter_idx in range(self.max_meters):
            # Extract the current meter's features: (batch_size, max_frames, n_components)
            meter_data = x[:, meter_idx, :, :]
            
            # Reshape for 2D CNN: (batch_size, 1, n_components, max_frames)
            # This treats n_components as height and max_frames as width
            meter_data = meter_data.transpose(1, 2).unsqueeze(1)
            
            # Pass through CNN layers
            for i in range(len(self.conv_layers)):
                meter_data = self.conv_layers[i](meter_data)
                meter_data = self.norm_layers[i](meter_data)
                meter_data = F.relu(meter_data)
                meter_data = self.pool_layers[i](meter_data)
            
            # Flatten and process with meter-level layers
            meter_flat = meter_data.reshape(batch_size, -1)
            meter_features = self.meter_layer(meter_flat)
            
            # Output prediction for this meter
            meter_output = torch.sigmoid(self.output_layer(meter_features))
            outputs.append(meter_output)
        
        # Stack all meter outputs
        stacked_outputs = torch.stack(outputs, dim=1)
        
        # Create mask for padding (where input is all zeros)
        mask = (torch.sum(torch.abs(x), dim=(2, 3)) > 0).float().unsqueeze(2)
        
        # Apply mask
        masked_outputs = stacked_outputs * mask
        
        return masked_outputs

Loss function

In [11]:
class BCEMaskedLoss(nn.Module):
    def __init__(self):
        super(BCEMaskedLoss, self).__init__()
        self.bce = nn.BCELoss(reduction='none')
    
    def forward(self, predictions, targets):
        # Create mask for valid positions (not -1)
        mask = (targets != -1).float()
        
        # Replace -1 with 0 to make it valid for BCE
        valid_targets = torch.clamp(targets, 0, 1)
        
        # Compute BCE loss
        loss = self.bce(predictions, valid_targets)
        
        # Apply mask and compute mean over valid positions
        masked_loss = loss * mask
        num_valid = torch.sum(mask)
        
        if num_valid > 0:
            return torch.sum(masked_loss) / num_valid
        else:
            return torch.tensor(0.0, device=predictions.device)

Model configuration with hyperparemeter tuning ranges

In [12]:
def get_config(tune_config=None):
    base_config = {
        "data": {
            "max_frames": max_frames_per_meter,
            "max_meters": max_meters
        },
        "audio": {
            "sr": 12000,
            "hop_length": 128,
            "n_components": 3
        },
        "model": {
            "dropout": 0.3,
            "cnn": {
                "channels": [32, 64],  # Fewer layers
                "kernel_sizes": [(3, 3), (3, 3)],
                # CRITICAL: Don't pool the feature dimension (first value = 1)
                "pool_sizes": [(1, 2), (1, 2)]  # Only pool time dimension
            }
        },
        "training": {
            "batch_size": 16,
            "learning_rate": 0.001,
            "epochs": 50,
            "weight_decay": 1e-5,
            "optimizer": "adam",
            "scheduler": "reduce_lr_on_plateau"
        }
    }
    
    # Override with tuning parameters if provided
    if tune_config:
        if "dropout" in tune_config:
            base_config["model"]["dropout"] = tune_config["dropout"]
        if "lr" in tune_config:
            base_config["training"]["learning_rate"] = tune_config["lr"]
        if "weight_decay" in tune_config:
            base_config["training"]["weight_decay"] = tune_config["weight_decay"]
        if "batch_size" in tune_config:
            base_config["training"]["batch_size"] = tune_config["batch_size"]
        if "optimizer" in tune_config:
            base_config["training"]["optimizer"] = tune_config["optimizer"]
        if "kernel_size" in tune_config:
            ks = tune_config["kernel_size"]
            base_config["model"]["cnn"]["kernel_sizes"] = [(ks, ks), (ks, ks)]
    
    return base_config

Shuffle and split dataset

In [13]:
# Find all audio files with corresponding labels
audio_files = []
for filename in os.listdir(audio_dir):
    if filename.endswith((".wav", ".mp3", ".flac")):
        song_id = os.path.splitext(filename)[0]
        label_path = os.path.join(labels_dir, f"{song_id}_labels_{experiment}_v{version}.pkl.gz")
        
        if os.path.exists(label_path):
            audio_files.append(os.path.join(audio_dir, filename))

# Set random seed for reproducibility
random_seed = 42
random.seed(random_seed)
np.random.seed(random_seed)
torch.manual_seed(random_seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(random_seed)
    torch.backends.cudnn.deterministic = True

# Shuffle and split dataset
np.random.shuffle(audio_files)
n_samples = len(audio_files)
train_split = 0.7
val_split = 0.15
test_split = 0.15
n_train = int(n_samples * train_split)
n_val = int(n_samples * val_split)

train_files = audio_files[:n_train]
val_files = audio_files[n_train:n_train+n_val]
test_files = audio_files[n_train+n_val:]

print(f"Train files: {len(train_files)}")
print(f"Validation files: {len(val_files)}")
print(f"Test files: {len(test_files)}")


Train files: 231
Validation files: 49
Test files: 51


Training function with Ray Tune for Hhyperparameter optimization

In [14]:
# Training function for Ray Tune
def train_tune(config, checkpoint_dir=None, train_files=None, val_files=None, 
               labels_dir=None, precomputed_dir=None, experiment=None, version=None):
    # Get model configuration with tuning parameters
    model_config = get_config(config)
    
    # Create datasets
    train_dataset = MelDecompDataset(
        train_files, labels_dir, model_config, precomputed_dir, 
        n_components=model_config["audio"]["n_components"],
        experiment=experiment, version=version
    )
    
    val_dataset = MelDecompDataset(
        val_files, labels_dir, model_config, precomputed_dir, 
        n_components=model_config["audio"]["n_components"],
        experiment=experiment, version=version
    )
    
    # Create data loaders
    batch_size = model_config["training"]["batch_size"]
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True, num_workers=0
    )
    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False, num_workers=0
    )
    
    # Set device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Create model
    model = MelDecompModel(model_config)
    model.to(device)
    
    # Setup loss function
    criterion = BCEMaskedLoss()
    
    # Setup optimizer based on config
    lr = model_config["training"]["learning_rate"]
    weight_decay = model_config["training"]["weight_decay"]
    
    if model_config["training"]["optimizer"] == "adam":
        optimizer = Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    elif model_config["training"]["optimizer"] == "sgd":
        optimizer = SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=weight_decay)
    elif model_config["training"]["optimizer"] == "adamw":
        optimizer = AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    else:
        optimizer = Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    # Setup scheduler
    scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)
    
    # Load checkpoint if provided
    if checkpoint_dir:
        checkpoint = os.path.join(checkpoint_dir, "checkpoint")
        model_state, optimizer_state = torch.load(checkpoint)
        model.load_state_dict(model_state)
        optimizer.load_state_dict(optimizer_state)
    
    # Training loop
    for epoch in range(10):  # Train for 10 epochs within Ray Tune
        model.train()
        train_loss = 0.0
        
        for i, (inputs, targets) in enumerate(train_loader):
            inputs, targets = inputs.to(device), targets.to(device)
            
            # Zero gradients
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(inputs)
            
            # Calculate loss
            loss = criterion(outputs, targets)
            
            # Backward pass and optimize
            loss.backward()
            optimizer.step()
            
            # Update statistics
            train_loss += loss.item()
        
        # Calculate average training loss
        train_loss /= len(train_loader)
        
        # Validate
        model.eval()
        val_loss = 0.0
        val_predictions = []
        val_targets = []
        
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                
                # Forward pass
                outputs = model(inputs)
                
                # Calculate loss
                loss = criterion(outputs, targets)
                
                # Update statistics
                val_loss += loss.item()
                
                # Store predictions and targets for metrics calculation
                mask = targets != -1
                val_predictions.extend(outputs[mask].cpu().numpy().flatten())
                val_targets.extend(targets[mask].cpu().numpy().flatten())
        
        # Calculate average validation loss
        val_loss /= len(val_loader)
        
        # Calculate metrics
        if val_predictions:
            binary_predictions = (np.array(val_predictions) > 0.5).astype(int)
            binary_targets = np.array(val_targets).astype(int)
            
            precision, recall, f1, _ = precision_recall_fscore_support(
                binary_targets, binary_predictions, average='binary'
            )
            accuracy = accuracy_score(binary_targets, binary_predictions)
            
            # Update scheduler
            scheduler.step(val_loss)
            
            # Report metrics to Ray Tune
            tune.report(
                loss=val_loss,
                accuracy=accuracy,
                f1=f1,
                precision=precision,
                recall=recall,
                train_loss=train_loss
            )
        
        # Save checkpoint for Ray Tune
        with tune.checkpoint_dir(epoch) as checkpoint_dir:
            path = os.path.join(checkpoint_dir, "checkpoint")
            torch.save((model.state_dict(), optimizer.state_dict()), path)


Standard training function

In [15]:
def train_model(train_loader, val_loader, model, criterion, optimizer, scheduler, device, 
               num_epochs, output_dir, config, tensorboard_dir=None):
    
    # Setup TensorBoard if directory provided
    writer = None
    if tensorboard_dir:
        writer = SummaryWriter(tensorboard_dir)
    
    # Track best model
    best_val_loss = float('inf')
    train_losses = []
    val_losses = []
    early_stop_patience = 10
    early_stop_counter = 0
    
    # Training loop
    print(f"Starting training for {num_epochs} epochs")
    for epoch in range(1, num_epochs + 1):
        print(f"\nEpoch {epoch}/{num_epochs}")
        
        # Train
        model.train()
        epoch_train_loss = 0.0
        
        for i, (inputs, targets) in enumerate(tqdm(train_loader, desc="Training")):
            inputs, targets = inputs.to(device), targets.to(device)
            
            # Zero gradients
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(inputs)
            
            # Calculate loss
            loss = criterion(outputs, targets)
            
            # Backward pass and optimize
            loss.backward()
            optimizer.step()
            
            # Update statistics
            epoch_train_loss += loss.item()
        
        # Calculate average training loss
        epoch_train_loss /= len(train_loader)
        train_losses.append(epoch_train_loss)
        print(f"Training Loss: {epoch_train_loss:.4f}")
        
        # Log to TensorBoard
        if writer:
            writer.add_scalar('Loss/train', epoch_train_loss, epoch)
        
        # Validate
        model.eval()
        epoch_val_loss = 0.0
        val_predictions = []
        val_targets = []
        
        with torch.no_grad():
            for inputs, targets in tqdm(val_loader, desc="Validation"):
                inputs, targets = inputs.to(device), targets.to(device)
                
                # Forward pass
                outputs = model(inputs)
                
                # Calculate loss
                loss = criterion(outputs, targets)
                
                # Update statistics
                epoch_val_loss += loss.item()
                
                # Store predictions and targets for metrics calculation
                mask = targets != -1
                val_predictions.extend(outputs[mask].cpu().numpy().flatten())
                val_targets.extend(targets[mask].cpu().numpy().flatten())
        
        # Calculate average validation loss
        epoch_val_loss /= len(val_loader)
        val_losses.append(epoch_val_loss)
        print(f"Validation Loss: {epoch_val_loss:.4f}")
        
        # Log to TensorBoard
        if writer:
            writer.add_scalar('Loss/validation', epoch_val_loss, epoch)
        
        # Calculate metrics
        if val_predictions:
            binary_predictions = (np.array(val_predictions) > 0.5).astype(int)
            binary_targets = np.array(val_targets).astype(int)
            
            precision, recall, f1, _ = precision_recall_fscore_support(
                binary_targets, binary_predictions, average='binary'
            )
            accuracy = accuracy_score(binary_targets, binary_predictions)
            
            print(f"Validation Metrics - Accuracy: {accuracy:.4f}, "
                  f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")
            
            # Log metrics to TensorBoard
            if writer:
                writer.add_scalar('Metrics/accuracy', accuracy, epoch)
                writer.add_scalar('Metrics/precision', precision, epoch)
                writer.add_scalar('Metrics/recall', recall, epoch)
                writer.add_scalar('Metrics/f1', f1, epoch)
                writer.add_scalar('Metrics/learning_rate', optimizer.param_groups[0]['lr'], epoch)
        
        # Update learning rate
        if isinstance(scheduler, ReduceLROnPlateau):
            scheduler.step(epoch_val_loss)
        else:
            scheduler.step()
        
        # Early stopping check
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            early_stop_counter = 0
            
            # Save model checkpoint
            model_path = os.path.join(output_dir, "best_model.pt")
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': epoch_val_loss,
                'train_loss': epoch_train_loss,
                'config': config
            }, model_path)
            print(f"Saved best model to {model_path}")
        else:
            early_stop_counter += 1
            print(f"EarlyStopping counter: {early_stop_counter} out of {early_stop_patience}")
            
            if early_stop_counter >= early_stop_patience:
                print("Early stopping triggered")
                break
    
    # Close TensorBoard writer
    if writer:
        writer.close()
    
    # Plot loss curves
    plt.figure(figsize=(10, 5))
    plt.plot(train_losses, label='Training Loss')
    plt.plot(val_losses, label='Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.title('Training and Validation Losses')
    plt.savefig(os.path.join(output_dir, 'loss_curves.png'))
    plt.show()
    
    return train_losses, val_losses

Run hyperparameter tuning

In [16]:
# Run hyperparameter tuning with Ray Tune
def run_hyperparameter_tuning(train_files, val_files, labels_dir, precomputed_dir,
                             experiment="spec", version="1", output_dir=None,
                             num_samples=10, max_num_epochs=10, gpus_per_trial=0.5):
    
    # Configure search space
    config = {
        "lr": tune.loguniform(1e-4, 1e-2),
        "batch_size": tune.choice([4, 8, 16, 32]),
        "dropout": tune.uniform(0.1, 0.5),
        "weight_decay": tune.loguniform(1e-6, 1e-3),
        "optimizer": tune.choice(["adam", "sgd", "adamw"]),
        "kernel_size": tune.choice([3, 5, 7])
    }
    
    # Create scheduler with metric and mode specified
    scheduler = ASHAScheduler(
        metric="loss",  # Keep these in the scheduler
        mode="min",     # Keep these in the scheduler
        max_t=max_num_epochs,
        grace_period=2,
        reduction_factor=2
    )
    
    # Setup reporter
    reporter = CLIReporter(
        parameter_columns=["lr", "batch_size", "dropout", "weight_decay", "optimizer", "kernel_size"],
        metric_columns=["loss", "accuracy", "f1", "training_iteration"]
    )
    
    # Wrap training function with fixed parameters
    train_fn = partial(
        train_tune,
        train_files=train_files,
        val_files=val_files,
        labels_dir=labels_dir, 
        precomputed_dir="../data/features/melspectrogram",
        experiment=experiment,
        version=version
    )
    
    # Initialize Ray
    if not ray.is_initialized():
        ray.init(ignore_reinit_error=True)
    
    # Run hyperparameter tuning
    result = tune.run(
        train_fn,
        resources_per_trial={"cpu": 2, "gpu": gpus_per_trial},
        config=config,
        num_samples=num_samples,
        scheduler=scheduler,
        progress_reporter=reporter,
        name="chorus_detection_tune",
        local_dir=output_dir if output_dir else "./ray_results"
        # Remove metric="loss" and mode="min" from here
    )
    
    # Get best trial
    best_trial = result.get_best_trial("loss", "min", "last")
    print(f"Best trial config: {best_trial.config}")
    print(f"Best trial final validation loss: {best_trial.last_result['loss']}")
    print(f"Best trial final validation accuracy: {best_trial.last_result['accuracy']}")
    print(f"Best trial final validation F1: {best_trial.last_result['f1']}")
    
    return best_trial.config

Create dataset and loaders

In [17]:
# Create datasets and train the model
config = get_config()

# Create datasets
train_dataset = MelDecompDataset(
    train_files, labels_dir, config, precomputed_dir, 
    n_components=config["audio"]["n_components"],
    experiment=experiment, version=version
)

val_dataset = MelDecompDataset(
    val_files, labels_dir, config, precomputed_dir, 
    n_components=config["audio"]["n_components"],
    experiment=experiment, version=version
)

test_dataset = MelDecompDataset(
    test_files, labels_dir, config, precomputed_dir, 
    n_components=config["audio"]["n_components"],
    experiment=experiment, version=version
)

# Create data loaders
batch_size = config["training"]["batch_size"]

train_loader = DataLoader(
    train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True
)

val_loader = DataLoader(
    val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True
)

test_loader = DataLoader(
    test_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True
)

In [18]:
def precompute_all_data(dataset, desc="Processing"):
    """Force processing of all items in dataset to create precomputed files"""
    print(f"Pre-computing {len(dataset)} files...")
    for i in tqdm(range(len(dataset)), desc=desc):
        try:
            _ = dataset[i]  # This will trigger processing and saving
        except Exception as e:
            print(f"Error processing item {i}: {e}")
    
    print("Pre-computation complete!")

# Pre-compute all datasets
precompute_all_data(train_dataset, "Training data")
precompute_all_data(val_dataset, "Validation data") 
precompute_all_data(test_dataset, "Test data")

# Check results
precomputed_files = os.listdir(precomputed_dir) if os.path.exists(precomputed_dir) else []
print(f"Total precomputed files created: {len(precomputed_files)}")

Pre-computing 231 files...


Training data:   0%|          | 0/231 [00:00<?, ?it/s]

⚠️  CONVERGENCE WARNING for song_id: 258
⚠️  CONVERGENCE WARNING for song_id: 418
⚠️  CONVERGENCE WARNING for song_id: 331
⚠️  CONVERGENCE WARNING for song_id: 465
⚠️  CONVERGENCE WARNING for song_id: 142
Pre-computation complete!
Pre-computing 49 files...


Validation data:   0%|          | 0/49 [00:00<?, ?it/s]

Pre-computation complete!
Pre-computing 51 files...


Test data:   0%|          | 0/51 [00:00<?, ?it/s]

⚠️  CONVERGENCE WARNING for song_id: 324
⚠️  CONVERGENCE WARNING for song_id: 389
Pre-computation complete!
Total precomputed files created: 331


Run hyperparameter tuning

In [41]:
best_config = run_hyperparameter_tuning(
    train_files, val_files, labels_dir, precomputed_dir,
    experiment=experiment, version=version, output_dir=output_dir,
    num_samples=5, max_num_epochs=5, gpus_per_trial=0.5 if torch.cuda.is_available() else 0
)
config = get_config(best_config)

2025-05-22 14:04:16,328	INFO tune.py:613 -- [output] This uses the legacy output and progress reporter, as Jupyter notebooks are not supported by the new engine, yet. For more information, please see https://github.com/ray-project/ray/issues/36949
2025-05-22 14:04:16,378	WARNING trial.py:648 -- The path to the trial log directory is too long (max length: 260. Consider using `trial_dirname_creator` to shorten the path. Path: C:\Users\denni\AppData\Local\Temp\ray\session_2025-05-22_12-26-04_406972_14112\artifacts\2025-05-22_14-04-16\chorus_detection_tune\driver_artifacts\train_tune_52250_00000_0_batch_size=4,dropout=0.4395,kernel_size=7,lr=0.0057,optimizer=sgd,weight_decay=0.0000_2025-05-22_14-04-16
2025-05-22 14:04:16,383	WARNING trial.py:648 -- The path to the trial log directory is too long (max length: 260. Consider using `trial_dirname_creator` to shorten the path. Path: C:\Users\denni\AppData\Local\Temp\ray\session_2025-05-22_12-26-04_406972_14112\artifacts\2025-05-22_14-04-16\chor

== Status ==
Current time: 2025-05-22 14:04:16 (running for 00:00:00.27)
Using AsyncHyperBand: num_stopped=0
Bracket: Iter 4.000: None | Iter 2.000: None
Logical resource usage: 10.0/16 CPUs, 0/1 GPUs (0.0/1.0 accelerator_type:G)
Result logdir: C:/Users/denni/AppData/Local/Temp/ray/session_2025-05-22_12-26-04_406972_14112/artifacts/2025-05-22_14-04-16/chorus_detection_tune/driver_artifacts
Number of trials: 5/5 (5 PENDING)


== Status ==
Current time: 2025-05-22 14:04:21 (running for 00:00:05.28)
Using AsyncHyperBand: num_stopped=0
Bracket: Iter 4.000: None | Iter 2.000: None
Logical resource usage: 10.0/16 CPUs, 0/1 GPUs (0.0/1.0 accelerator_type:G)
Result logdir: C:/Users/denni/AppData/Local/Temp/ray/session_2025-05-22_12-26-04_406972_14112/artifacts/2025-05-22_14-04-16/chorus_detection_tune/driver_artifacts
Number of trials: 5/5 (5 PENDING)




2025-05-22 14:04:25,048	WARNING trial.py:648 -- The path to the trial log directory is too long (max length: 260. Consider using `trial_dirname_creator` to shorten the path. Path: C:\Users\denni\AppData\Local\Temp\ray\session_2025-05-22_12-26-04_406972_14112\artifacts\2025-05-22_14-04-16\chorus_detection_tune\driver_artifacts\train_tune_52250_00003_3_batch_size=8,dropout=0.2754,kernel_size=3,lr=0.0006,optimizer=adamw,weight_decay=0.0001_2025-05-22_14-04-16
2025-05-22 14:04:25,050	WARNING trial.py:648 -- The path to the trial log directory is too long (max length: 260. Consider using `trial_dirname_creator` to shorten the path. Path: C:\Users\denni\AppData\Local\Temp\ray\session_2025-05-22_12-26-04_406972_14112\artifacts\2025-05-22_14-04-16\chorus_detection_tune\driver_artifacts\train_tune_52250_00003_3_batch_size=8,dropout=0.2754,kernel_size=3,lr=0.0006,optimizer=adamw,weight_decay=0.0001_2025-05-22_14-04-16
2025-05-22 14:04:25,057	WARNING trial.py:648 -- The path to the trial log dire

Trial name
train_tune_52250_00000
train_tune_52250_00001
train_tune_52250_00002
train_tune_52250_00003
train_tune_52250_00004


2025-05-22 14:04:25,290	ERROR tune_controller.py:1332 -- Trial task failed for trial train_tune_52250_00003
Traceback (most recent call last):
  File "c:\Users\denni\anaconda3\envs\chorus-detection\lib\site-packages\ray\air\execution\_internal\event_manager.py", line 110, in resolve_future
    result = ray.get(future)
  File "c:\Users\denni\anaconda3\envs\chorus-detection\lib\site-packages\ray\_private\auto_init_hook.py", line 21, in auto_init_wrapper
    return fn(*args, **kwargs)
  File "c:\Users\denni\anaconda3\envs\chorus-detection\lib\site-packages\ray\_private\client_mode_hook.py", line 103, in wrapper
    return func(*args, **kwargs)
  File "c:\Users\denni\anaconda3\envs\chorus-detection\lib\site-packages\ray\_private\worker.py", line 2667, in get
    values, debugger_breakpoint = worker.get_objects(object_refs, timeout=timeout)
  File "c:\Users\denni\anaconda3\envs\chorus-detection\lib\site-packages\ray\_private\worker.py", line 864, in get_objects
    raise value.as_instanceof

== Status ==
Current time: 2025-05-22 14:04:25 (running for 00:00:09.00)
Using AsyncHyperBand: num_stopped=0
Bracket: Iter 4.000: None | Iter 2.000: None
Logical resource usage: 4.0/16 CPUs, 0/1 GPUs (0.0/1.0 accelerator_type:G)
Result logdir: C:/Users/denni/AppData/Local/Temp/ray/session_2025-05-22_12-26-04_406972_14112/artifacts/2025-05-22_14-04-16/chorus_detection_tune/driver_artifacts
Number of trials: 5/5 (5 ERROR)
+------------------------+----------+-----------------+-------------+--------------+-----------+----------------+-------------+---------------+
| Trial name             | status   | loc             |          lr |   batch_size |   dropout |   weight_decay | optimizer   |   kernel_size |
|------------------------+----------+-----------------+-------------+--------------+-----------+----------------+-------------+---------------|
| train_tune_52250_00000 | ERROR    | 127.0.0.1:30308 | 0.00568521  |            4 |  0.439468 |    2.56959e-06 | sgd         |             7 |


TuneError: ('Trials did not complete', [train_tune_52250_00000, train_tune_52250_00001, train_tune_52250_00002, train_tune_52250_00003, train_tune_52250_00004])

## Model Training

## Model Training

## Model Training

## Model Training

## Model Training

In [ ]:
def train_mel_decomp_model(config, data_config, output_dir):
    """Train the mel decomposition model."""
    # Setup device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    # Create data loaders
    print("Creating data loaders...")
    train_loader, val_loader, test_loader = create_mel_decomp_data_loaders(
        config=config,
        audio_dir=data_config["audio_dir"],
        labels_dir=data_config["labels_dir"],
        precomputed_dir=data_config.get("precomputed_dir"),
        train_split=data_config.get("train_split", 0.7),
        val_split=data_config.get("val_split", 0.15),
        test_split=data_config.get("test_split", 0.15),
        n_components=config["audio"]["n_components"]
    )
    
    # Create model
    print("Creating model...")
    model = MelDecompModel(config)
    model.to(device)
    
    # Setup loss function, optimizer and scheduler
    criterion = BCEMaskedLoss()
    optimizer = Adam(
        model.parameters(),
        lr=config["training"]["learning_rate"],
        weight_decay=config["training"]["weight_decay"]
    )
    scheduler = ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=0.5,
        patience=5,
        verbose=True
    )
    
    # Training parameters
    epochs = config["training"]["epochs"]
    best_val_loss = float('inf')
    train_losses = []
    val_losses = []
    
    # Training loop
    print(f"Starting training for {epochs} epochs")
    for epoch in range(1, epochs + 1):
        print(f"\nEpoch {epoch}/{epochs}")
        
        # Train
        model.train()
        epoch_train_loss = 0.0
        
        for i, (inputs, targets) in enumerate(train_loader):
            inputs, targets = inputs.to(device), targets.to(device)
            
            # Zero gradients
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(inputs)
            
            # Calculate loss
            loss = criterion(outputs, targets)
            
            # Backward pass and optimize
            loss.backward()
            optimizer.step()
            
            # Update statistics
            epoch_train_loss += loss.item()
            
            # Print batch progress
            if (i + 1) % 10 == 0 or i == len(train_loader) - 1:
                print(f"Batch {i+1}/{len(train_loader)}, Loss: {loss.item():.4f}")
        
        # Calculate average training loss
        epoch_train_loss /= len(train_loader)
        train_losses.append(epoch_train_loss)
        print(f"Training Loss: {epoch_train_loss:.4f}")
        
        # Validate
        model.eval()
        epoch_val_loss = 0.0
        val_predictions = []
        val_targets = []
        
        with torch.no_grad():
            for inputs, targets in val_loader:
                inputs, targets = inputs.to(device), targets.to(device)
                
                # Forward pass
                outputs = model(inputs)
                
                # Calculate loss
                loss = criterion(outputs, targets)
                
                # Update statistics
                epoch_val_loss += loss.item()
                
                # Store predictions and targets for metrics calculation
                mask = targets != -1
                val_predictions.extend(outputs[mask].cpu().numpy().flatten())
                val_targets.extend(targets[mask].cpu().numpy().flatten())
        
        # Calculate average validation loss
        epoch_val_loss /= len(val_loader)
        val_losses.append(epoch_val_loss)
        print(f"Validation Loss: {epoch_val_loss:.4f}")
        
        # Calculate metrics
        if val_predictions:
            binary_predictions = (np.array(val_predictions) > 0.5).astype(int)
            binary_targets = np.array(val_targets).astype(int)
            
            precision, recall, f1, _ = precision_recall_fscore_support(
                binary_targets, binary_predictions, average='binary'
            )
            accuracy = accuracy_score(binary_targets, binary_predictions)
            
            print(f"Validation Metrics - Accuracy: {accuracy:.4f}, "
                  f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")
        
        # Update learning rate
        scheduler.step(epoch_val_loss)
        
        # Save model if it's the best so far
        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            
            # Save model checkpoint
            model_path = os.path.join(output_dir, "best_model.pt")
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': epoch_val_loss,
                'train_loss': epoch_train_loss,
                'config': config
            }, model_path)
            print(f"Saved best model to {model_path}")
    
    # Evaluate on test set
    print("\nEvaluating on test set...")
    model.eval()
    test_loss = 0.0
    test_predictions = []
    test_targets = []
    
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            
            # Forward pass
            outputs = model(inputs)
            
            # Calculate loss
            loss = criterion(outputs, targets)
            
            # Update statistics
            test_loss += loss.item()
            
            # Store predictions and targets for metrics calculation
            mask = targets != -1
            test_predictions.extend(outputs[mask].cpu().numpy().flatten())
            test_targets.extend(targets[mask].cpu().numpy().flatten())
    
    # Calculate average test loss
    test_loss /= len(test_loader)
    print(f"Test Loss: {test_loss:.4f}")
    
    # Calculate test metrics
    if test_predictions:
        binary_predictions = (np.array(test_predictions) > 0.5).astype(int)
        binary_targets = np.array(test_targets).astype(int)
        
        precision, recall, f1, _ = precision_recall_fscore_support(
            binary_targets, binary_predictions, average='binary'
        )
        accuracy = accuracy_score(binary_targets, binary_predictions)
        
        print(f"Test Metrics - Accuracy: {accuracy:.4f}, "
              f"Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")
    
    # Save final model
    final_model_path = os.path.join(output_dir, "final_model.pt")
    torch.save({
        'epoch': epochs,
        'model_state_dict': model.state_dict(),
        'test_loss': test_loss,
        'config': config
    }, final_model_path)
    print(f"Saved final model to {final_model_path}")
    
    # Save training history and metrics
    history = {
        'train_losses': train_losses,
        'val_losses': val_losses,
        'best_val_loss': best_val_loss,
        'test_loss': test_loss,
        'test_metrics': {
            'accuracy': float(accuracy),
            'precision': float(precision),
            'recall': float(recall),
            'f1': float(f1)
        }
    }
    
    with open(os.path.join(output_dir, "training_history.json"), "w") as f:
        import json
        json.dump(history, f, indent=4)
    
    print("Training completed!")
    return model

## Model Evaluation

In [ ]:
# Create a visualization function to see model predictions
def visualize_predictions(model, data_loader, device, num_samples=3):
    """Visualize model predictions against ground truth."""
    model.eval()
    
    samples_processed = 0
    
    with torch.no_grad():
        for inputs, targets in data_loader:
            inputs, targets = inputs.to(device), targets.to(device)
            
            # Forward pass
            outputs = model(inputs)
            
            # Process each sample in the batch
            for i in range(inputs.shape[0]):
                if samples_processed >= num_samples:
                    return
                
                # Get predictions and ground truth for this sample
                sample_outputs = outputs[i].cpu().numpy().flatten()
                sample_targets = targets[i].cpu().numpy().flatten()
                
                # Create mask for valid positions
                valid_mask = sample_targets != -1
                valid_outputs = sample_outputs[valid_mask]
                valid_targets = sample_targets[valid_mask]
                
                if len(valid_outputs) == 0:
                    continue
                
                # Plot
                plt.figure(figsize=(15, 5))
                
                plt.subplot(1, 1, 1)
                plt.plot(valid_outputs, label='Predictions')
                plt.plot(valid_targets, label='Ground Truth')
                plt.axhline(y=0.5, color='r', linestyle='-', alpha=0.3, label='Threshold')
                plt.ylim(-0.1, 1.1)
                plt.legend()
                plt.title(f'Chorus Predictions vs Ground Truth (Sample {samples_processed+1})')
                plt.tight_layout()
                plt.show()
                
                samples_processed += 1


# Train and evaluate the model
if __name__ == "__main__":
    # Create configuration
    config = create_mel_decomp_config()
    
    # Data configuration
    data_config = {
        "audio_dir": audio_dir,
        "labels_dir": labels_dir,
        "precomputed_dir": precomputed_dir,
        "train_split": 0.7,
        "val_split": 0.15,
        "test_split": 0.15
    }
    
    # Train model
    model = train_mel_decomp_model(config, data_config, output_dir)
    
    # Visualize predictions
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    test_loader = create_mel_decomp_data_loaders(
        config=config,
        audio_dir=data_config["audio_dir"],
        labels_dir=data_config["labels_dir"],
        precomputed_dir=data_config.get("precomputed_dir"),
        train_split=0,  # Only test set
        val_split=0,
        test_split=1.0,
        n_components=config["audio"]["n_components"]
    )[2]  # Get test loader
    
    visualize_predictions(model, test_loader, device, num_samples=3)